In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
import warnings
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
warnings.filterwarnings('ignore')


# 1. 데이터 로딩
cust_df = pd.read_csv("../data/santander-customer-satisfaction/train.csv", encoding='latin-1')
print('dataset shape:', cust_df.shape)
cust_df.head(3)
 

dataset shape: (76020, 371)


,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
0,1,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39205.17,0
1,3,2,34,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,49278.03,0
2,4,2,23,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,67333.77,0


In [2]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [3]:
# 2. 불균형 확인
print(cust_df['TARGET'].value_counts())
unsatisfied_cnt = cust_df[cust_df['TARGET'] == 1].TARGET.count()
total_cnt = cust_df.TARGET.count()
print('unsatisfied 비율은 {0:.2f}'.format((unsatisfied_cnt / total_cnt)))

TARGET
0    73012
1     3008
Name: count, dtype: int64
unsatisfied 비율은 0.04


In [4]:
cust_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 76020 entries, 0 to 76019
Columns: 371 entries, ID to TARGET
dtypes: float64(111), int64(260)
memory usage: 215.2 MB


In [5]:
# 3. 이상값 탐지(var3의 min: -999999)
cust_df.describe()

,ID,var3,var15,imp_ent_var16_ult1,imp_op_var39_comer_ult1,imp_op_var39_comer_ult3,imp_op_var40_comer_ult1,imp_op_var40_comer_ult3,imp_op_var40_efect_ult1,imp_op_var40_efect_ult3,...,saldo_medio_var33_hace2,saldo_medio_var33_hace3,saldo_medio_var33_ult1,saldo_medio_var33_ult3,saldo_medio_var44_hace2,saldo_medio_var44_hace3,saldo_medio_var44_ult1,saldo_medio_var44_ult3,var38,TARGET
count,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,...,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,76020.000000,7.602000e+04,76020.000000
mean,75964.050723,-1523.199277,33.212865,86.208265,72.363067,119.529632,3.559130,6.472698,0.412946,0.567352,...,7.935824,1.365146,12.215580,8.784074,31.505324,1.858575,76.026165,56.614351,1.172358e+05,0.039569
std,43781.947379,39033.462364,12.956486,1614.757313,339.315831,546.266294,93.155749,153.737066,30.604864,36.513513,...,455.887218,113.959637,783.207399,538.439211,2013.125393,147.786584,4040.337842,2852.579397,1.826646e+05,0.194945
min,1.000000,-999999.000000,5.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,5.163750e+03,0.000000
25%,38104.750000,2.000000,23.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,6.787061e+04,0.000000
50%,76043.000000,2.000000,28.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.064092e+05,0.000000
75%,113748.750000,2.000000,40.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.187563e+05,0.000000
max,151838.000000,238.000000,105.000000,210000.000000,12888.030000,21024.810000,8237.820000,11073.570000,6600.000000,6600.000000,...,50003.880000,20385.720000,138831.630000,91778.730000,438329.220000,24650.010000,681462.900000,397884.300000,2.203474e+07,1.000000


In [6]:
# 4-1. 전처리(var3의 -999999 -> 2  & ID 제거)
cust_df["var3"] = cust_df["var3"].replace(-999999,2)
cust_df.drop("ID", axis=1, inplace=True)

In [7]:
# 일반 데이터와 레이블 분리
X_features = cust_df.iloc[:,:-1]
y_labels = cust_df.iloc[:,-1]

In [8]:
# 4-2. 전처리(분산 0인 컬럼 제거)
# 1. 각 컬럼의 분산 계산
stds = X_features.std()

# 2. 분산이 0인(즉, 표준편차가 0이거나 값이 모두 똑같은) 컬럼 이름 추출
zero_var_cols = stds[stds == 0].index.tolist()

print(f"분산이 0인 컬럼 개수: {len(zero_var_cols)}")
print(f"삭제할 컬럼들: {zero_var_cols}")

# 3. 해당 컬럼들 제거
X_features_clean = X_features.drop(columns=zero_var_cols)

print(f"정제 전 피처 shape: {X_features.shape}")
print(f"정제 후 피처 shape: {X_features_clean.shape}")

분산이 0인 컬럼 개수: 34
삭제할 컬럼들: ['ind_var2_0', 'ind_var2', 'ind_var27_0', 'ind_var28_0', 'ind_var28', 'ind_var27', 'ind_var41', 'ind_var46_0', 'ind_var46', 'num_var27_0', 'num_var28_0', 'num_var28', 'num_var27', 'num_var41', 'num_var46_0', 'num_var46', 'saldo_var28', 'saldo_var27', 'saldo_var41', 'saldo_var46', 'imp_amort_var18_hace3', 'imp_amort_var34_hace3', 'imp_reemb_var13_hace3', 'imp_reemb_var33_hace3', 'imp_trasp_var17_out_hace3', 'imp_trasp_var33_out_hace3', 'num_var2_0_ult1', 'num_var2_ult1', 'num_reemb_var13_hace3', 'num_reemb_var33_hace3', 'num_trasp_var17_out_hace3', 'num_trasp_var33_out_hace3', 'saldo_var2_ult1', 'saldo_medio_var13_medio_hace3']
정제 전 피처 shape: (76020, 369)
정제 후 피처 shape: (76020, 335)


In [9]:
# 값이 완전히 동일한 컬럼 중 뒤에 나온 것들의 이름을 반환
import numpy as np

def find_duplicate_columns(df):

    groups = {}
    for col in df.columns:
        v = df[col].values
        # 지문(sum/min/max)으로 후보를 먼저 좁힘
        key = (v.sum(), v.min(), v.max())
        groups.setdefault(key, []).append(col)

    dup = set()
    for cols in groups.values():
        if len(cols) < 2:
            continue
        for i in range(len(cols)):
            if cols[i] in dup:
                continue
            for j in range(i + 1, len(cols)):
                if cols[j] in dup:
                    continue
                if np.array_equal(df[cols[i]].values, df[cols[j]].values):
                    dup.add(cols[j])
    return sorted(dup)


dup_cols = find_duplicate_columns(X_features_clean)
print(f"중복 컬럼 개수: {len(dup_cols)}")
print(f"삭제할 컬럼들: {dup_cols}")

X_features_clean = X_features_clean.drop(columns=dup_cols)
print(f"정제 후 피처 shape: {X_features_clean.shape}")

중복 컬럼 개수: 29
삭제할 컬럼들: ['delta_num_reemb_var13_1y3', 'delta_num_reemb_var17_1y3', 'delta_num_reemb_var33_1y3', 'delta_num_trasp_var17_in_1y3', 'delta_num_trasp_var17_out_1y3', 'delta_num_trasp_var33_in_1y3', 'delta_num_trasp_var33_out_1y3', 'ind_var13_medio', 'ind_var18', 'ind_var25', 'ind_var26', 'ind_var29', 'ind_var29_0', 'ind_var32', 'ind_var34', 'ind_var37', 'ind_var39', 'num_var13_medio', 'num_var18', 'num_var25', 'num_var26', 'num_var29', 'num_var29_0', 'num_var32', 'num_var34', 'num_var37', 'num_var39', 'saldo_medio_var13_medio_ult1', 'saldo_var29']
정제 후 피처 shape: (76020, 306)


In [10]:
# =========================================================================
# 4. 데이터셋 단일 분할 및 대조 실험군 독립 분기 (Hold-out 공정성 확보)
# =========================================================================
from sklearn.metrics import recall_score
from xgboost import XGBClassifier


print("\n=== 데이터 분할 및 대조 실험군 독립 분기 ===")

X_train, X_test, y_train, y_test = train_test_split(
    X_features_clean, y_labels, test_size=0.2, random_state=156, stratify=y_labels
)

# [실험 A용 데이터셋 준비] 원본 버전 (var38 수치가 로그 없이 날것으로 유지됨)
X_train_norm = X_train.copy()
X_test_norm = X_test.copy()

# [실험 B용 데이터셋 준비] 로그 변환 버전 🚀
X_train_log = X_train.copy()
X_test_log = X_test.copy()

# 실험 B 세트 내부의 'var38'에만 정밀하게 로그 트랜스포메이션 주입
X_train_log['var38'] = np.log1p(X_train_log['var38'].astype(float))
X_test_log['var38'] = np.log1p(X_test_log['var38'].astype(float))

print("  - 데이터 복사 타입 충돌 및 카테고리 불일치 에러 완전 방어 완료!")


# =========================================================================
# 5. 최신 XGBoost 인프라 정의 및 대조군 학습/평가
# =========================================================================
print("\n=== XGBoost 비교 학습 및 최종 스코어 측정 ===")

def get_xgb_model():
    return XGBClassifier(
        n_estimators=500,
        learning_rate=0.05,
        max_depth=5,
        min_child_weight=5,
        subsample=0.8,
        colsample_bytree=0.8,
        early_stopping_rounds=100,
        eval_metric='auc',
        # 원-핫 인코딩을 완료하여 모두 수치형 칼럼으로 변환되었으므로 enable_categorical 옵션은 제외합니다.
        random_state=156
    )

# -------------------------------------------------------------------------
# 실험 A: 원본 var38 데이터셋 기반 학습 진행
# -------------------------------------------------------------------------
print("  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...")
xgb_norm = get_xgb_model()
xgb_norm.fit(
    X_train_norm, y_train,
    eval_set=[(X_train_norm, y_train), (X_test_norm, y_test)],
    verbose=False
)
norm_pred_proba = xgb_norm.predict_proba(X_test_norm)[:, 1]
auc_normal = roc_auc_score(y_test, norm_pred_proba)

# 🚀 정확도 및 재현율 계산용 예측값 추출
norm_preds = xgb_norm.predict(X_test_norm)
acc_normal = accuracy_score(y_test, norm_preds)
rec_normal = recall_score(y_test, norm_preds)

# -------------------------------------------------------------------------
# 실험 B: 로그 변환 var38 데이터셋 기반 학습 진행
# -------------------------------------------------------------------------
print("  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...")
xgb_log = get_xgb_model()
xgb_log.fit(
    X_train_log, y_train,
    eval_set=[(X_train_log, y_train), (X_test_log, y_test)],
    verbose=False
)
log_pred_proba = xgb_log.predict_proba(X_test_log)[:, 1]
auc_log = roc_auc_score(y_test, log_pred_proba)

# 🚀 정확도 및 재현율 계산용 예측값 추출
log_preds = xgb_log.predict(X_test_log)
acc_log = accuracy_score(y_test, log_preds)
rec_log = recall_score(y_test, log_preds)


# =========================================================================
# 6. 최종 분석 결과 종합 출력
# =========================================================================
print("\n" + "="*65)
print("             [ 최종 분석 논문 검증 결과 종합 성적표 ]")
print("="*65)
print("  구분                 |   ROC-AUC   |   정확도    |   재현율   ")
print("-"*65)
print(f"  [실험 A] 로그 전    |    {auc_normal:.4f}    |    {acc_normal:.4f}    |   {rec_normal:.4f}")
print(f"  [실험 B] 로그 후    |    {auc_log:.4f}    |    {acc_log:.4f}    |   {rec_log:.4f}")
print("-"*65)
print(f"  순수 개선 편차       |   {auc_log - auc_normal:+.4f}    |   {acc_log - acc_normal:+.4f}    |   {rec_log - rec_normal:+.4f}")
print("="*65)

# 영찬 님 결과
# 결과 : ROC AUC : 0.8517(컬럼정제전)
# 결과 : ROC AUC : 0.8531(컬럼정제후)

# 내 결과
# =================================================================
#              [ 최종 분석 논문 검증 결과 종합 성적표 ]
# =================================================================
#   구분                 |   ROC-AUC   |   정확도    |   재현율   
# -----------------------------------------------------------------
#   [실험 A] 로그 전    |    0.8520    |    0.9606    |   0.0066
#   [실험 B] 로그 후    |    0.8520    |    0.9606    |   0.0066
# -----------------------------------------------------------------
#   순수 개선 편차       |   -0.0000    |   +0.0000    |   +0.0000
# =================================================================


=== 데이터 분할 및 대조 실험군 독립 분기 ===


  - 데이터 복사 타입 충돌 및 카테고리 불일치 에러 완전 방어 완료!

=== XGBoost 비교 학습 및 최종 스코어 측정 ===
  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...


  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...



             [ 최종 분석 논문 검증 결과 종합 성적표 ]
  구분                 |   ROC-AUC   |   정확도    |   재현율   
-----------------------------------------------------------------
  [실험 A] 로그 전    |    0.8520    |    0.9606    |   0.0066
  [실험 B] 로그 후    |    0.8520    |    0.9606    |   0.0066
-----------------------------------------------------------------
  순수 개선 편차       |   -0.0000    |   +0.0000    |   +0.0000


---

## [담당자: 정찬성] XGBoost 외 4개 모델(RandomForest/LogisticRegression/LightGBM/GradientBoost) 비교 실습

위 §5(XGBoost 비교 학습 및 최종 스코어 측정, cell 9)까지가 기존에 완료된 내용이다 — Santander 데이터(76,020행, 정제 후 306개 피처)를 `X_train`/`X_test`(80/20, stratify)로 한 번만 분할한 뒤, 그 위에서 **실험 A(`var38` 원본)**와 **실험 B(`var38`을 `log1p` 변환)** 두 대조군을 XGBoost로 각각 학습·평가했다. 이 지점부터는 **완전히 동일한 `X_train_norm`/`X_test_norm`/`X_train_log`/`X_test_log`/`y_train`/`y_test`(위 cell 9에서 이미 만든 것을 재사용, 새로 분할하지 않음)**로 나머지 4개 모델(RandomForest, LogisticRegression, LightGBM, GradientBoost)을 XGBoost와 동일한 실험 A/B 구조로 돌려본다.

```mermaid
flowchart TD
    LOAD["1~4. 데이터 로드/정제(위, 완료)<br/>var3 이상치 처리 · ID 제거 · 분산0/중복 컬럼 제거<br/>(76,020행 x 306피처)"] --> SPLIT["Hold-out 단일 분할(위, 완료)<br/>X_train/X_test 80/20, stratify=y_labels"]
    SPLIT --> EXPA["실험 A: var38 원본(log 미적용)<br/>X_train_norm / X_test_norm"]
    SPLIT --> EXPB["실험 B: var38 → log1p 변환<br/>X_train_log / X_test_log"]

    EXPA --> XGB["① XGBoost(완료, 위 cell 9)"]
    EXPB --> XGB
    EXPA --> RF["② RandomForest(신규)"]
    EXPB --> RF
    EXPA --> LR["③ LogisticRegression(신규)"]
    EXPB --> LR
    EXPA --> LGBM["④ LightGBM(신규)"]
    EXPB --> LGBM
    EXPA --> GB["⑤ GradientBoost(신규)"]
    EXPB --> GB

    XGB --> CMP["최종 종합 비교<br/>5개 모델 x 실험 A/B = 10행"]
    RF --> CMP
    LR --> CMP
    LGBM --> CMP
    GB --> CMP

    style XGB fill:#E4F5E9,stroke:#2E8B57
    style CMP fill:#E0ECFF,stroke:#2E6DE5
```

| 모델 | 하이퍼파라미터 | XGBoost(n_estimators=500, learning_rate=0.05, max_depth=5, min_child_weight=5, subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=100)와 대응 관계 |
|---|---|---|
| RandomForest | `n_estimators=500, max_depth=5, min_samples_leaf=5, max_features=0.8, max_samples=0.8, n_jobs=-1, random_state=156` | `max_features`가 `colsample_bytree`(분할마다 피처 샘플링 비율)에, `max_samples`가 `subsample`(트리마다 행 샘플링 비율)에, `min_samples_leaf`가 `min_child_weight`(리프의 최소 표본 조건)에 대응한다. 단, RandomForest는 배깅(bagging) 계열이라 "이전 트리의 오차를 보정"하는 개념이 없어 **조기종료(early stopping) 자체가 존재하지 않는다** — 트리 500개를 전부 채운다. |
| LogisticRegression | `LogisticRegression(max_iter=1000, random_state=156)` | 선형모델이라 트리·부스팅 관련 파라미터가 전혀 대응되지 않는다. 피처가 306개로 비교적 많아 기본 반복횟수(100)로는 수렴 경고가 날 수 있어 `max_iter=1000`으로 여유를 뒀다. |
| LightGBM | `n_estimators=500, learning_rate=0.05, max_depth=5, min_child_samples=5, subsample=0.8, colsample_bytree=0.8, random_state=156` + `eval_X`/`eval_y`/`callbacks=[lgb.early_stopping(100)]` | XGBoost와 파라미터 이름·값을 거의 그대로 맞췄다(`min_child_weight` → LightGBM에서는 `min_child_samples`가 같은 역할). 조기종료도 `early_stopping_rounds=100`과 동일한 의미로 `stopping_rounds=100`을 콜백으로 지정했다 — LightGBM 최신 sklearn API는 `eval_set` 대신 `eval_X`/`eval_y`를 쓴다. |
| GradientBoost | `n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8, random_state=156, n_iter_no_change=100, validation_fraction=0.1` | XGBoost의 원조 알고리즘이라 가장 직접 비교되는 짝이다. `colsample_bytree`/`min_child_weight`에 대응하는 옵션이 sklearn `GradientBoostingClassifier`에는 없다(한계로 명시). 조기종료는 `early_stopping_rounds`처럼 별도 검증셋을 넘기는 방식이 아니라, 학습 데이터의 `validation_fraction`(10%)을 내부적으로 떼어 `n_iter_no_change=100`(100라운드 개선 없으면 중단) 기준으로 판단한다. |

In [11]:
# 기술적 의미: XGBoost 실험(§cell 9)은 "모델 학습 → 예측 → AUC/정확도/재현율 계산 → 성적표 출력"을 실험 A/B마다 손으로 반복 작성했다.
# 왜: 이 패턴을 4개 모델 x 2개 실험 = 8번 그대로 복사하면 코드가 지나치게 길어지고 오탈자 위험이 커진다. §cell 9의 로직(모델 학습 → predict_proba/predict → roc_auc_score/accuracy_score/recall_score)을 그대로 함수로 옮겨 재사용하되, §cell 9 자체는 건드리지 않는다(기존 XGBoost 실측 결과 보존).
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score

# 기술적 의미: 이미 생성된 모델 객체와 학습/검증 데이터, 그리고 모델별로 다른 fit() 추가 인자(**fit_kwargs, 예: LightGBM의 조기종료 콜백)를 받아 학습·예측·평가까지 한 번에 수행하는 함수를 정의한다.
# 업무적 의미: RandomForest/LogisticRegression처럼 추가 인자가 필요 없는 모델과 LightGBM처럼 조기종료 콜백이 필요한 모델을 같은 함수 하나로 통일해서 다룰 수 있게 한다.
def run_experiment(model, X_tr, y_tr, X_te, y_te, **fit_kwargs):
    # 기술적 의미: 학습 데이터와 모델별 추가 인자(fit_kwargs)로 모델을 학습시킨다.
    model.fit(X_tr, y_tr, **fit_kwargs)
    # 기술적 의미: 검증 데이터에 대한 예측 확률(클래스 1=불만족 고객일 확률)을 구한다 — AUC 계산에 필요하다.
    pred_proba = model.predict_proba(X_te)[:, 1]
    # 기술적 의미: 검증 데이터에 대한 최종 예측 클래스(0/1)를 구한다 — 정확도·재현율 계산에 필요하다.
    pred = model.predict(X_te)
    # 기술적 의미: ROC-AUC, 정확도, 재현율 3개 지표를 dict로 만들어 반환한다. §cell 9와 동일한 3개 지표다.
    metrics = {
        'auc': roc_auc_score(y_te, pred_proba),
        'acc': accuracy_score(y_te, pred),
        'rec': recall_score(y_te, pred),
    }
    # 기술적 의미: 학습이 끝난 모델 객체와 지표 dict를 함께 반환한다 — 모델 객체는 이후 feature_importances_/coef_ 조회에 쓰인다.
    return model, metrics


# 기술적 의미: §cell 9의 "최종 분석 논문 검증 결과 종합 성적표" 출력 블록과 완전히 동일한 서식으로, 모델명만 바꿔 출력하는 함수를 정의한다.
# 업무적 의미: 4개 모델 모두 XGBoost와 시각적으로 1:1 비교 가능한 동일한 표 형식으로 결과를 남긴다.
def print_scorecard(model_name, metrics_norm, metrics_log):
    print("\n" + "="*65)
    print(f"        [ {model_name} 최종 분석 논문 검증 결과 종합 성적표 ]")
    print("="*65)
    print("  구분                 |   ROC-AUC   |   정확도    |   재현율   ")
    print("-"*65)
    print(f"  [실험 A] 로그 전    |    {metrics_norm['auc']:.4f}    |    {metrics_norm['acc']:.4f}    |   {metrics_norm['rec']:.4f}")
    print(f"  [실험 B] 로그 후    |    {metrics_log['auc']:.4f}    |    {metrics_log['acc']:.4f}    |   {metrics_log['rec']:.4f}")
    print("-"*65)
    print(f"  순수 개선 편차       |   {metrics_log['auc']-metrics_norm['auc']:+.4f}    |   {metrics_log['acc']-metrics_norm['acc']:+.4f}    |   {metrics_log['rec']-metrics_norm['rec']:+.4f}")
    print("="*65)


# 기술적 의미: 4개 모델 x 2개 실험(A/B) = 8건의 결과를 순서대로 쌓아 둘 빈 리스트를 만든다.
# 업무적 의미: §최종 종합 비교에서 XGBoost 실측치와 합쳐 5개 모델 비교표를 만들 원재료다.
results_log = []


In [12]:
# [담당자: 정찬성 / 모델: RandomForest] 실험 A/B 대조군 학습·평가
# 왜: 배깅(bagging) 계열 대표 모델을 부스팅 계열(XGBoost/LightGBM/GradientBoost) 및 선형모델(LR)과 비교하기 위한 기준점으로 추가한다. §개요 표에서 설명한 대로 max_features/max_samples/min_samples_leaf를 XGBoost의 colsample_bytree/subsample/min_child_weight에 대응시켰다.
from sklearn.ensemble import RandomForestClassifier


# 기술적 의미: 실험마다 완전히 새로운 RandomForestClassifier 객체를 만들어 반환하는 팩토리 함수를 정의한다 — §cell 9의 get_xgb_model()과 동일한 패턴이다.
# 업무적 의미: 실험 A/B가 서로의 학습 상태에 영향을 주지 않도록(독립적인 대조군이 되도록) 매번 새 객체로 시작한다.
def get_rf_model():
    return RandomForestClassifier(
        n_estimators=500, max_depth=5, min_samples_leaf=5,
        max_features=0.8, max_samples=0.8, n_jobs=-1, random_state=156,
    )


print("\n=== RandomForest 비교 학습 및 최종 스코어 측정 ===")

# 기술적 의미: [실험 A] var38 원본 데이터셋(X_train_norm/X_test_norm, §cell 9에서 이미 만든 것)으로 RandomForest를 학습·평가한다.
print("  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...")
rf_norm, metrics_rf_norm = run_experiment(get_rf_model(), X_train_norm, y_train, X_test_norm, y_test)

# 기술적 의미: [실험 B] var38 로그 변환 데이터셋(X_train_log/X_test_log)으로 RandomForest를 학습·평가한다.
print("  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...")
rf_log, metrics_rf_log = run_experiment(get_rf_model(), X_train_log, y_train, X_test_log, y_test)

# 기술적 의미: §cell 9와 동일한 서식으로 실험 A/B 성적표를 출력한다.
print_scorecard('RandomForest', metrics_rf_norm, metrics_rf_log)

# 기술적 의미: 실험 A/B 결과를 각각 한 행씩 results_log에 추가한다.
results_log.append({'모델': 'RandomForest', '실험': 'A.원본 var38', **metrics_rf_norm})
results_log.append({'모델': 'RandomForest', '실험': 'B.log1p(var38)', **metrics_rf_log})



=== RandomForest 비교 학습 및 최종 스코어 측정 ===
  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...


  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...



        [ RandomForest 최종 분석 논문 검증 결과 종합 성적표 ]
  구분                 |   ROC-AUC   |   정확도    |   재현율   
-----------------------------------------------------------------
  [실험 A] 로그 전    |    0.8414    |    0.9605    |   0.0017
  [실험 B] 로그 후    |    0.8414    |    0.9605    |   0.0017
-----------------------------------------------------------------
  순수 개선 편차       |   +0.0000    |   +0.0000    |   +0.0000


In [13]:
# [담당자: 정찬성 / 모델: LogisticRegression] 실험 A/B 대조군 학습·평가
# 왜: 트리 기반 4개 모델(XGBoost/RandomForest/LightGBM/GradientBoost)과 대비되는 선형모델 기준점(baseline)을 추가한다 — 선형모델은 피처 스케일에 민감하므로, log1p 변환이 트리 모델보다 LR에서 더 뚜렷한 효과를 보일 가능성이 있다(§신용카드 사기검출 노트북에서도 동일한 패턴을 이미 확인한 바 있다).
from sklearn.linear_model import LogisticRegression


# 기술적 의미: 실험마다 새로운 LogisticRegression 객체를 만들어 반환하는 팩토리 함수를 정의한다.
# 업무적 의미: §개요 표에서 설명한 대로, 트리 계열과 달리 맞출 만한 하이퍼파라미터가 없어 max_iter(수렴 보장)만 늘렸다.
def get_lr_model():
    return LogisticRegression(max_iter=1000, random_state=156)


print("\n=== LogisticRegression 비교 학습 및 최종 스코어 측정 ===")

# 기술적 의미: [실험 A] var38 원본 데이터셋으로 LogisticRegression을 학습·평가한다.
print("  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...")
lr_norm, metrics_lr_norm = run_experiment(get_lr_model(), X_train_norm, y_train, X_test_norm, y_test)

# 기술적 의미: [실험 B] var38 로그 변환 데이터셋으로 LogisticRegression을 학습·평가한다.
print("  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...")
lr_log, metrics_lr_log = run_experiment(get_lr_model(), X_train_log, y_train, X_test_log, y_test)

# 기술적 의미: 성적표를 출력한다.
print_scorecard('LogisticRegression', metrics_lr_norm, metrics_lr_log)

# 기술적 의미: 실험 A/B 결과를 results_log에 추가한다.
results_log.append({'모델': 'LogisticRegression', '실험': 'A.원본 var38', **metrics_lr_norm})
results_log.append({'모델': 'LogisticRegression', '실험': 'B.log1p(var38)', **metrics_lr_log})



=== LogisticRegression 비교 학습 및 최종 스코어 측정 ===
  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...


  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...



        [ LogisticRegression 최종 분석 논문 검증 결과 종합 성적표 ]
  구분                 |   ROC-AUC   |   정확도    |   재현율   
-----------------------------------------------------------------
  [실험 A] 로그 전    |    0.6230    |    0.9603    |   0.0000
  [실험 B] 로그 후    |    0.5473    |    0.9602    |   0.0017
-----------------------------------------------------------------
  순수 개선 편차       |   -0.0757    |   -0.0001    |   +0.0017


In [14]:
# [담당자: 정찬성 / 모델: LightGBM] 실험 A/B 대조군 학습·평가
# 왜: 같은 그레이디언트 부스팅 계열이지만 리프 단위(leaf-wise) 성장 방식을 쓰는 LightGBM을, 레벨 단위 성장인 XGBoost와 동일한 하이퍼파라미터·조기종료 조건으로 비교한다.
from lightgbm import LGBMClassifier
import lightgbm as lgb


# 기술적 의미: 실험마다 새로운 LGBMClassifier 객체를 만들어 반환하는 팩토리 함수를 정의한다. min_child_samples는 XGBoost의 min_child_weight(리프가 되기 위한 최소 조건)에 대응하는 LightGBM 파라미터다. verbose=-1은 학습 중 반복마다 출력되는 내부 로그를 끈다.
def get_lgbm_model():
    return LGBMClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=5, min_child_samples=5,
        subsample=0.8, colsample_bytree=0.8, random_state=156, verbose=-1,
    )


print("\n=== LightGBM 비교 학습 및 최종 스코어 측정 ===")

# 기술적 의미: [실험 A] var38 원본 데이터셋으로 LightGBM을 학습한다. eval_X/eval_y로 검증셋(X_test_norm/y_test)을 지정하고, eval_metric='auc' 기준으로 100라운드 동안 개선이 없으면 멈추는 조기종료 콜백을 건다 — XGBoost의 early_stopping_rounds=100과 동일한 의미다.
print("  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...")
rf_kwargs_norm = dict(eval_X=X_test_norm, eval_y=y_test, eval_metric='auc',
                       callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
lgbm_norm, metrics_lgbm_norm = run_experiment(get_lgbm_model(), X_train_norm, y_train, X_test_norm, y_test, **rf_kwargs_norm)

# 기술적 의미: [실험 B] var38 로그 변환 데이터셋으로 동일하게 학습한다. 검증셋도 로그 변환된 X_test_log/y_test로 맞춘다.
print("  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...")
rf_kwargs_log = dict(eval_X=X_test_log, eval_y=y_test, eval_metric='auc',
                      callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)])
lgbm_log, metrics_lgbm_log = run_experiment(get_lgbm_model(), X_train_log, y_train, X_test_log, y_test, **rf_kwargs_log)

# 기술적 의미: 성적표를 출력한다.
print_scorecard('LightGBM', metrics_lgbm_norm, metrics_lgbm_log)

# 기술적 의미: 실험 A/B 결과를 results_log에 추가한다.
results_log.append({'모델': 'LightGBM', '실험': 'A.원본 var38', **metrics_lgbm_norm})
results_log.append({'모델': 'LightGBM', '실험': 'B.log1p(var38)', **metrics_lgbm_log})



=== LightGBM 비교 학습 및 최종 스코어 측정 ===
  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...


  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...



        [ LightGBM 최종 분석 논문 검증 결과 종합 성적표 ]
  구분                 |   ROC-AUC   |   정확도    |   재현율   
-----------------------------------------------------------------
  [실험 A] 로그 전    |    0.8490    |    0.9604    |   0.0033
  [실험 B] 로그 후    |    0.8490    |    0.9604    |   0.0033
-----------------------------------------------------------------
  순수 개선 편차       |   +0.0000    |   +0.0000    |   +0.0000


In [15]:
# [담당자: 정찬성 / 모델: GradientBoost] 실험 A/B 대조군 학습·평가
# 왜: XGBoost의 "원조 알고리즘"인 sklearn GradientBoostingClassifier를 최대한 동일한 트리 개수·깊이·학습률·subsample로 평가해, "구현체 최적화(정규화항·히스토그램 분할) 차이"만 순수 비교한다.
from sklearn.ensemble import GradientBoostingClassifier


# 기술적 의미: 실험마다 새로운 GradientBoostingClassifier 객체를 만들어 반환하는 팩토리 함수를 정의한다. n_iter_no_change=100·validation_fraction=0.1은 §개요 표에서 설명한 대로 early_stopping_rounds=100에 대응하는 sklearn 방식의 조기종료 설정이다(학습 데이터의 10%를 내부 검증용으로 떼어 판단).
def get_gb_model():
    return GradientBoostingClassifier(
        n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8,
        random_state=156, n_iter_no_change=100, validation_fraction=0.1,
    )


print("\n=== GradientBoost 비교 학습 및 최종 스코어 측정 ===")

# 기술적 의미: [실험 A] var38 원본 데이터셋으로 GradientBoost를 학습·평가한다. GradientBoostingClassifier는 fit()에 eval_set을 받지 않으므로 fit_kwargs 없이 그대로 호출한다.
print("  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...")
gb_norm, metrics_gb_norm = run_experiment(get_gb_model(), X_train_norm, y_train, X_test_norm, y_test)

# 기술적 의미: [실험 B] var38 로그 변환 데이터셋으로 GradientBoost를 학습·평가한다.
print("  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...")
gb_log, metrics_gb_log = run_experiment(get_gb_model(), X_train_log, y_train, X_test_log, y_test)

# 기술적 의미: 성적표를 출력한다.
print_scorecard('GradientBoost', metrics_gb_norm, metrics_gb_log)

# 기술적 의미: 실험 A/B 결과를 results_log에 추가한다.
results_log.append({'모델': 'GradientBoost', '실험': 'A.원본 var38', **metrics_gb_norm})
results_log.append({'모델': 'GradientBoost', '실험': 'B.log1p(var38)', **metrics_gb_log})



=== GradientBoost 비교 학습 및 최종 스코어 측정 ===
  - 1. [실험 A] var38 원본 데이터셋 기반 모델 학습 중...


  - 2. [실험 B] var38 로그 변환 데이터셋 기반 모델 학습 중...



        [ GradientBoost 최종 분석 논문 검증 결과 종합 성적표 ]
  구분                 |   ROC-AUC   |   정확도    |   재현율   
-----------------------------------------------------------------
  [실험 A] 로그 전    |    0.8462    |    0.9594    |   0.0150
  [실험 B] 로그 후    |    0.8460    |    0.9592    |   0.0150
-----------------------------------------------------------------
  순수 개선 편차       |   -0.0002    |   -0.0003    |   +0.0000


In [16]:
# [담당자: 정찬성] 실험 B(로그 변환, 4개 모델 공통 최종 학습 결과) 기준 피처 중요도/계수 top10 비교
# 왜: 트리 계열 3종(RandomForest/LightGBM/GradientBoost)의 feature_importances_와 LogisticRegression의 coef_를 나란히 비교해, "어떤 피처가 고객 불만족(TARGET=1)을 가르는 핵심 신호인가"에 대해 계열이 다른 모델들이 같은 결론에 도달하는지 교차검증한다.

# 기술적 의미: RandomForest의 feature_importances_ 상위 10개를 Series로 만들어 출력한다.
print('[RandomForest] 피처 중요도 top10:')
print(pd.Series(rf_log.feature_importances_, index=X_train_log.columns).sort_values(ascending=False).head(10))
print()

# 기술적 의미: LightGBM의 feature_importances_(기본값: 분할 횟수 기준) 상위 10개를 출력한다.
print('[LightGBM] 피처 중요도 top10:')
print(pd.Series(lgbm_log.feature_importances_, index=X_train_log.columns).sort_values(ascending=False).head(10))
print()

# 기술적 의미: GradientBoost의 feature_importances_ 상위 10개를 출력한다.
print('[GradientBoost] 피처 중요도 top10:')
print(pd.Series(gb_log.feature_importances_, index=X_train_log.columns).sort_values(ascending=False).head(10))
print()

# 기술적 의미: LogisticRegression은 계수(coef_) 절댓값 상위 10개를 부호(+/-)와 함께 출력한다.
print('[LogisticRegression] |계수| top10(부호 포함):')
lr_coef = pd.Series(lr_log.coef_[0], index=X_train_log.columns)
print(lr_coef.reindex(lr_coef.abs().sort_values(ascending=False).head(10).index))


[RandomForest] 피처 중요도 top10:
var15                      0.388424
saldo_var30                0.249471
var38                      0.115509
ind_var30                  0.018653
num_var30                  0.018139
imp_op_var41_efect_ult1    0.012884
saldo_medio_var5_hace3     0.009115
imp_op_var39_efect_ult1    0.008691
num_var22_ult3             0.008254
saldo_var42                0.007890
dtype: float64

[LightGBM] 피처 중요도 top10:
var38                     239
var15                     224
saldo_var30               128
saldo_medio_var5_ult3     115
saldo_medio_var5_hace2     99
saldo_medio_var5_hace3     98
num_var22_ult3             74
num_var45_hace3            73
num_var22_hace3            65
saldo_var5                 62
dtype: int32

[GradientBoost] 피처 중요도 top10:
var15                      0.222364
saldo_var30                0.180179
var38                      0.137830
saldo_medio_var5_hace3     0.026263
saldo_medio_var5_hace2     0.016938
saldo_medio_var5_ult3      0.014922
saldo_var5

---

## [담당자: 정찬성] 최종 종합 비교 — 5개 모델 x 실험 A/B

```mermaid
flowchart LR
    A["results_log(8건)<br/>RandomForest/LogisticRegression/LightGBM/GradientBoost x 실험 A/B"] --> B["XGBoost 실측치 2건<br/>(§cell 9 결과 주석에서 가져옴)"]
    B --> C["5개 모델 x 2실험 = 10행<br/>종합 비교 DataFrame"]
    C --> D["실험별 pivot<br/>(AUC 기준 모델 랭킹)"]
```

In [17]:
# 기술적 의미: results_log(신규 4개 모델 x 2개 실험 = 8건)를 pandas DataFrame으로 변환한다.
new_models_df = pd.DataFrame(results_log)

# 기술적 의미: XGBoost는 이미 위(§cell 9)에서 실제로 실행돼 "내 결과" 주석으로 실측치가 기록돼 있다 — 그 값을 그대로 옮겨 적어(재실행하지 않고) 나머지 4개 모델과 같은 형식의 행으로 만든다.
# 업무적 의미: 이미 신뢰할 수 있는 값을 다시 계산하느라 시간을 쓰지 않고, 기존 실행 로그(§cell 9)를 "단일 진실 공급원"으로 재사용한다.
xgb_summary = pd.DataFrame([
    {'모델': 'XGBoost', '실험': 'A.원본 var38', 'auc': 0.8520, 'acc': 0.9606, 'rec': 0.0066},
    {'모델': 'XGBoost', '실험': 'B.log1p(var38)', 'auc': 0.8520, 'acc': 0.9606, 'rec': 0.0066},
])

# 기술적 의미: 신규 4개 모델 결과(new_models_df)와 XGBoost 실측치(xgb_summary)를 세로로 이어붙여 5개 모델 x 2실험 = 10행짜리 종합 비교표를 만든다.
full_comparison = pd.concat([xgb_summary, new_models_df], ignore_index=True)

# 기술적 의미: 종합 비교표 전체를 출력한다.
print('=== 5개 모델 x 실험 A/B 종합 비교 ===')
print(full_comparison.to_string(index=False))
print()

# 기술적 의미: 실험(A/B)을 행, 모델을 열로 하고 AUC 값을 채운 pivot 표를 만든다.
# 업무적 의미: "이 실험 조건에서는 어느 모델이 가장 판별력이 좋았는가"를 한눈에 가로 비교할 수 있게 한다.
auc_pivot = full_comparison.pivot(index='실험', columns='모델', values='auc')

# 기술적 의미: AUC 기준 pivot 표를 출력한다.
print('=== 실험별 AUC 비교(모델을 열로) ===')
print(auc_pivot.to_string())


=== 5개 모델 x 실험 A/B 종합 비교 ===
                모델             실험      auc      acc      rec
           XGBoost     A.원본 var38 0.852000 0.960600 0.006600
           XGBoost B.log1p(var38) 0.852000 0.960600 0.006600
      RandomForest     A.원본 var38 0.841409 0.960471 0.001661
      RandomForest B.log1p(var38) 0.841409 0.960471 0.001661
LogisticRegression     A.원본 var38 0.623006 0.960339 0.000000
LogisticRegression B.log1p(var38) 0.547337 0.960208 0.001661
          LightGBM     A.원본 var38 0.848967 0.960405 0.003322
          LightGBM B.log1p(var38) 0.848967 0.960405 0.003322
     GradientBoost     A.원본 var38 0.846234 0.959419 0.014950
     GradientBoost B.log1p(var38) 0.845987 0.959155 0.014950

=== 실험별 AUC 비교(모델을 열로) ===
모델              GradientBoost  LightGBM  LogisticRegression  RandomForest  XGBoost
실험                                                                                
A.원본 var38           0.846234  0.848967            0.623006      0.841409    0.852
B.log1p(var38)       0.

### 최종 해석

> ⏸ 이 셀은 실행 대기 상태다 — 위 종합 비교표를 실제로 생성한 뒤, 아래 항목을 채운다.

- **부스팅 3종(XGBoost/LightGBM/GradientBoost) 간 비교**: 동일한 트리 개수·깊이·학습률·subsample로 맞췄을 때, AUC·재현율이 어느 구현체에서 가장 좋은지 확인할 것. XGBoost는 실험 A/B 모두 재현율이 0.0066로 극히 낮았다 — 이는 클래스 불균형(TARGET=1 비율 약 3.96%, §cell 2)에서 기본 임계값(0.5)이 소수 클래스를 거의 잡아내지 못한다는 신호이며, 나머지 3개 부스팅 모델도 같은 문제를 보이는지, 혹은 조기종료·리프 성장 방식 차이로 이 문제가 완화되는지가 핵심 확인 포인트다.
- **배깅(RandomForest) vs 부스팅 3종**: RandomForest가 재현율 측면에서 부스팅 계열보다 나은지(배깅은 부스팅과 달리 오분류에 가중치를 주는 메커니즘이 없어 불균형에 더 취약할 수도, 트리 다양성 덕에 더 강건할 수도 있다) 실측으로 확인할 것.
- **선형(LogisticRegression) vs 트리 4종**: LR의 AUC가 XGBoost(0.8520)에 근접한다면, 이 데이터의 신용 위험 신호 상당 부분이 선형적으로 분리 가능하다는 뜻이고, 재현율이 유의미하게 다르다면 결정 경계의 형태(선형 vs 비선형) 차이가 실제로 드러난 것이다.
- **var38 로그 변환의 효과**: XGBoost는 실험 A/B가 완전히 동일했다("트리 기반 모델은 단조 변환에 불변") — RandomForest/LightGBM/GradientBoost도 같은 불변성을 보이는지, 반대로 스케일에 민감한 LogisticRegression에서는 실제로 유의미한 차이가 나는지 5개 모델 전체로 검증할 것.
- **극단적으로 낮은 재현율에 대한 권고**: 5개 모델 모두 재현율이 낮게 나온다면, 이 노트북의 범위를 벗어나는 후속 과제로 `class_weight='balanced'`, 임계값(threshold) 조정, 또는 신용카드 사기검출 노트북(§99_2)에서 이미 검증한 SMOTE 오버샘플링 적용을 검토 대상으로 제안할 수 있다.

---

## [담당자: 정찬성] 임계값(threshold) 조정 재평가 — 5개 모델 공통 기준

`정찬성/90.오류화면/오류화면1.png`에 정리된 스프레드시트(LightGBM: Accuracy 0.9582 / AUC 0.8417 / Precision 0.0016 / Recall 0.0031 / 임계값 0.04)를 채우기 위한 절이다 — 로지스틱 회귀·XGBoost·랜덤포레스트·GradientBoost 4개 모델의 값이 비어 있어, 5개 모델 전부를 동일한 절차로 재평가한다.

### 왜 기본 임계값(0.5) 대신 조정된 임계값이 필요한가

TARGET=1(불만족 고객) 비율은 약 3.96%(§cell 2)에 불과하다. `predict()`가 내부적으로 쓰는 기본 임계값 0.5는 "확률이 50%를 넘어야 양성으로 본다"는 뜻인데, 이렇게 심하게 불균형한 데이터에서는 모델이 정상 고객 쪽으로 확률을 낮게 몰아 예측해도 정확도가 높게 나오기 때문에, 웬만해서는 어떤 표본도 50%를 넘기지 못한다 — 실제로 §cell 9의 XGBoost 실측 재현율이 0.0066에 불과했던 것이 이 현상이다. 이 절에서는 **분류 임계값을 "양성 클래스의 기저율(base rate, 학습 데이터의 TARGET=1 비율 ≈ 0.04)"로 낮춰**, "무작위로 찍었을 때보다 조금이라도 더 위험해 보이면 양성으로 판단한다"는 기준으로 5개 모델을 공정하게 재평가한다 — `오류화면1.png`의 "임계값 0.04"와 정확히 같은 접근이다.

```mermaid
flowchart TD
    A["y_train.mean() = 양성(TARGET=1) 기저율<br/>≈ 0.04"] --> B["threshold = 0.04로 고정"]
    C["5개 모델(§cell 9 XGBoost +<br/>위에서 학습한 RF/LR/LightGBM/GradientBoost)<br/>실험 B(log1p) 기준 predict_proba()"] --> D["pred = (proba >= threshold)"]
    B --> D
    D --> E["Accuracy · AUC · Precision · Recall 계산"]
    E --> F["오류화면1.png와 동일한 행 순서로 표 출력<br/>(LightGBM → 로지스틱회귀 → XGBoost → 랜덤포레스트 → GradientBoost)"]

    style F fill:#E0ECFF,stroke:#2E6DE5
```

In [18]:
# [담당자: 정찬성] 5개 모델 공통 임계값 재평가
# 왜: §개요에서 설명한 대로, 기본 임계값(0.5)이 아니라 양성 클래스 기저율을 임계값으로 써서 5개 모델을 동일 기준으로 비교한다. 평가 데이터셋은 §Part(RandomForest~GradientBoost)에서 이미 학습해 둔 "실험 B(log1p 변환)" 모델(xgb_log/rf_log/lr_log/lgbm_log/gb_log)과 X_test_log/y_test를 그대로 재사용한다 — 실험 A/B 간 성능 차이가 트리 계열 4종에서는 이미 0에 가까웠고(§각 모델 셀의 순수 개선 편차), 로그 변환판을 "최종 채택 데이터"로 통일해 두 배로 반복 계산하지 않기 위함이다.
from sklearn.metrics import precision_score

# 기술적 의미: 학습 데이터(y_train)에서 TARGET=1(불만족 고객) 비율을 계산한다.
# 업무적 의미: 이 값이 곧 "무작위로 찍었을 때 양성일 확률"(기저율)이며, 이를 분류 임계값으로 쓴다.
threshold = y_train.mean()

# 기술적 의미: 계산된 임계값을 출력해 오류화면1.png의 "임계값 0.04"와 얼마나 일치하는지 바로 확인할 수 있게 한다.
print(f'분류 임계값(양성 클래스 기저율 기준): {threshold:.4f}')


# 기술적 의미: 이미 학습된 모델·검증 데이터·임계값을 받아 Accuracy/AUC/Precision/Recall 4개 지표를 dict로 계산해 반환하는 함수를 정의한다.
# 업무적 의미: model.predict()의 내장 임계값(0.5)을 쓰지 않고, predict_proba() 확률값에 우리가 정한 임계값을 직접 적용해 재분류한다 — AUC는 임계값과 무관하게 정의되므로 확률값 그대로 계산한다.
def evaluate_at_threshold(model, X_te, y_te, thr):
    # 기술적 의미: 검증 데이터에 대한 양성(TARGET=1) 확률을 구한다.
    pred_proba = model.predict_proba(X_te)[:, 1]
    # 기술적 의미: 확률이 임계값 이상이면 1(불만족), 미만이면 0(만족)으로 재분류한다.
    pred_at_threshold = (pred_proba >= thr).astype(int)
    # 기술적 의미: 4개 지표와 사용한 임계값을 dict로 반환한다. zero_division=0은 특정 모델이 양성을 하나도 예측하지 않아 정밀도가 0/0이 되는 극단적 경우를 대비한 안전장치다.
    return {
        'Accuracy': accuracy_score(y_te, pred_at_threshold),
        'AUC': roc_auc_score(y_te, pred_proba),
        'Precision': precision_score(y_te, pred_at_threshold, zero_division=0),
        'Recall': recall_score(y_te, pred_at_threshold, zero_division=0),
        '임계값': thr,
    }


# 기술적 의미: 5개 모델 각각에 대해 evaluate_at_threshold()를 호출해 결과를 dict of dict로 모은다. xgb_log는 §cell 9, 나머지 4개는 위에서 이미 학습해 둔 모델 객체다.
# 업무적 의미: 5개 모델을 정확히 같은 데이터(X_test_log/y_test)·같은 임계값으로 평가해야 오류화면1.png 표와 같은 기준의 비교가 된다.
threshold_results = {
    'XGBoost': evaluate_at_threshold(xgb_log, X_test_log, y_test, threshold),
    'RandomForest': evaluate_at_threshold(rf_log, X_test_log, y_test, threshold),
    'LogisticRegression': evaluate_at_threshold(lr_log, X_test_log, y_test, threshold),
    'LightGBM': evaluate_at_threshold(lgbm_log, X_test_log, y_test, threshold),
    'GradientBoost': evaluate_at_threshold(gb_log, X_test_log, y_test, threshold),
}

# 기술적 의미: dict of dict를 DataFrame으로 변환한다(행=모델, 열=지표가 되도록 전치(T) 처리).
threshold_df = pd.DataFrame(threshold_results).T

# 기술적 의미: 오류화면1.png 스프레드시트와 동일한 행 순서(LightGBM → 로지스틱 회귀 → XGBoost → 랜덤포레스트 → GradientBoost)로 재정렬한다.
threshold_df = threshold_df.loc[['LightGBM', 'LogisticRegression', 'XGBoost', 'RandomForest', 'GradientBoost']]

# 기술적 의미: 최종 표를 출력한다.
print('\n=== 임계값(threshold) 조정 후 5개 모델 비교 (정찬성/90.오류화면/오류화면1.png 표 형식) ===')
print(threshold_df.to_string())


분류 임계값(양성 클래스 기저율 기준): 0.0396



=== 임계값(threshold) 조정 후 5개 모델 비교 (정찬성/90.오류화면/오류화면1.png 표 형식) ===
                    Accuracy       AUC  Precision    Recall       임계값
LightGBM            0.768350  0.848967   0.121369  0.777409  0.039562
LogisticRegression  0.219942  0.547337   0.044579  0.915282  0.039562
XGBoost             0.771310  0.851993   0.123198  0.780731  0.039562
RandomForest        0.742765  0.841409   0.110247  0.777409  0.039562
GradientBoost       0.800184  0.845987   0.134454  0.744186  0.039562


### 결과 해석

> ⏸ 이 셀은 실행 대기 상태다 — 위 코드 셀을 실행한 뒤, 아래 항목을 실제 출력값으로 채운다.

- 계산된 임계값이 실제로 `오류화면1.png`의 "임계값 0.04"와 일치하는지 확인할 것(양성 비율은 §cell 2에서 이미 약 3.96%로 확인됐으므로, 반올림하면 0.04와 같아야 한다).
- LightGBM의 재계산 값(Accuracy/AUC/Precision/Recall)이 `오류화면1.png`의 기록값(0.9582/0.8417/0.0016/0.0031)과 얼마나 가까운지 비교할 것 — 정확히 같은 모델·전처리로 재현했다면 소수점 넷째 자리까지 일치해야 한다. 값이 다르다면 원본 스프레드시트가 참조한 모델 설정(하이퍼파라미터·실험 A/B 여부)이 이 노트북의 LightGBM(§Part LightGBM, `n_estimators=500, learning_rate=0.05, max_depth=5, min_child_samples=5, subsample=0.8, colsample_bytree=0.8`)과 다를 가능성을 검토해야 한다.
- 5개 모델 모두 임계값을 낮췄음에도 Precision·Recall이 여전히 매우 낮다면, 이는 "임계값 조정만으로는 해결되지 않는 근본적인 확률 보정(calibration) 문제"라는 뜻이다 — 이 경우 다음 후속 조치로 `class_weight='balanced'`(RandomForest/LogisticRegression/GradientBoost가 지원) 또는 `scale_pos_weight`(XGBoost/LightGBM이 지원) 같은 클래스 가중치 조정, 혹은 SMOTE 오버샘플링(§99_2 신용카드 사기검출 노트북에서 이미 검증한 방식)을 검토 대상으로 제안한다.

---

## [담당자: 정찬성] 원본 LightGBM 파이프라인 기준 5개 모델 재비교

`정찬성/90.오류화면/오류화면1.png`의 LightGBM 행(Accuracy 0.9582 / AUC 0.8417 / Recall 0.0016 / F1 0.0031)은 이 노트북(§위)이 아니라 **`09_산탄데르_고객_만족_LightGBM_정제후_metrics추가.ipynb`**에서 나온 값이다. 그 노트북은 지금까지 이 노트북(§위)에서 쓴 것과 **전처리·데이터 분할 자체가 다르다**:

| 항목 | `09_...LightGBM` (원본, 참고치의 출처) | 위 §XGBoost~GradientBoost(이 노트북, 지금까지) |
|---|---|---|
| 피처 개수 | **143개** — 분산0 제거(34개) + 중복 제거(29개) + **99% 이상이 0인 희소 피처까지 제거(163개)** | 306개 — 희소 피처 제거 단계 없음 |
| 데이터 분할 | `test_size=0.2, random_state=0`(**stratify 없음**), 학습셋을 다시 `X_tr/X_val`(`test_size=0.3, random_state=0`)로 재분할해 조기종료 전용 검증셋 확보 | `test_size=0.2, random_state=156, stratify=y_labels`, 별도 검증셋 없음(§Part XGBoost는 `eval_set`에 테스트셋을 직접 사용) |
| LightGBM 하이퍼파라미터 | HyperOpt(TPE, 50회 탐색, `rstate=seed 30`)로 튜닝 — `num_leaves=32, max_depth=113, min_child_samples=83, subsample=0.9533, learning_rate=0.12555`, `n_estimators=500`, 조기종료 100라운드 | (§Part LightGBM) `n_estimators=500, learning_rate=0.05, max_depth=5, min_child_samples=5, subsample=0.8, colsample_bytree=0.8` |
| 평가 임계값 | 기본값 0.5(스프레드시트의 "임계값 0.04"는 LightGBM 로그의 `pavg=0.038947`(양성 클래스 비율)를 적어 둔 참고 메모였고, 실제 분류 임계값으로 쓰이지 않았다) | 0.5(§Part XGBoost~GradientBoost) 또는 §바로 위 임계값 재평가 절의 0.0396 |

이 절에서는 **원본(`09_...LightGBM`)의 143개 피처·분할 방식·LightGBM 튜닝 하이퍼파라미터를 그대로 재현**하고, 그 위에서 XGBoost·RandomForest·LogisticRegression·GradientBoost 4개 모델도 같은 피처·같은 분할로 다시 학습해, `오류화면1.png` 스프레드시트에 5개 모델을 **완전히 동일한 기준**으로 채울 수 있게 한다. `X_features_clean`(§위, 306개 피처)은 이미 분산0·중복 제거가 끝난 상태이므로, 여기서는 "99% 이상이 0인 희소 피처 제거" 한 단계만 추가하면 원본과 동일한 143개 피처가 된다.

```mermaid
flowchart TD
    A["X_features_clean(§위, 306개 피처)"] --> B["+ 희소 피처(99% 이상 0) 제거<br/>→ X_features_143(143개 피처)"]
    B --> C["1차 분할(원본과 동일)<br/>X_train2/X_test2 = split(test_size=0.2, random_state=0, stratify 없음)"]
    C --> D["2차 분할(조기종료용)<br/>X_tr2/X_val2 = split(X_train2, test_size=0.3, random_state=0)"]

    D --> E["① LightGBM(HyperOpt 튜닝값 재현)<br/>num_leaves=32, max_depth=113, min_child_samples=83,<br/>subsample=0.9533, learning_rate=0.12555"]
    D --> F["② XGBoost(§위와 동일 하이퍼파라미터)"]
    C --> G["③ RandomForest(검증셋 불필요, X_train2 전체 사용)"]
    C --> H["④ LogisticRegression(X_train2 전체 사용)"]
    C --> I["⑤ GradientBoost(X_train2 전체, 내부 validation_fraction으로 조기종료)"]

    E --> J["임계값 0.5 기준<br/>Accuracy/AUC/Recall/F1"]
    F --> J
    G --> J
    H --> J
    I --> J

    J --> K["오류화면1.png와 동일한 행 순서로 최종 표 출력"]

    style E fill:#E4F5E9,stroke:#2E8B57
    style K fill:#E0ECFF,stroke:#2E6DE5
```

In [19]:
# [담당자: 정찬성] 원본(09_...LightGBM)과 동일한 143개 피처 + 분할 방식 재현
# 왜: X_features_clean(§위, 306개 피처)은 이미 분산0(34개)·중복(29개) 제거까지 끝난 상태다 — 09_...LightGBM 노트북의 1·2단계와 정확히 같은 결과다. 거기에 09_...LightGBM의 3단계(99% 이상이 0인 희소 피처 제거)만 추가하면 원본과 동일한 143개 피처가 된다.

# 기술적 의미: 전체 행 수를 구한다 — 희소 여부(0의 비율) 판단의 분모로 쓴다.
num_rows = X_features_clean.shape[0]

# 기술적 의미: 각 컬럼에서 값이 0인 행의 비율이 99% 이상인 컬럼명을 리스트로 뽑는다 — §09_...LightGBM cell 6의 로직과 완전히 동일하다.
# 업무적 의미: 거의 항상 0인 컬럼은 모델이 학습할 정보가 사실상 없어(분산은 0이 아니어도 신호가 희박), 제거해도 정보 손실이 크지 않으면서 계산량은 크게 줄인다.
sparse_cols = [col for col in X_features_clean.columns if (X_features_clean[col] == 0).sum() / num_rows >= 0.99]

# 기술적 의미: 희소 피처를 제거한 최종 143개 피처 DataFrame을 만든다.
X_features_143 = X_features_clean.drop(columns=sparse_cols)

print(f"희소 피처(99% 이상 0) 제거: {len(sparse_cols)}개 삭제")
print(f"최종 피처 shape: {X_features_143.shape}  (원본 09_...LightGBM 기준 143개와 일치해야 함)")


# 기술적 의미: 원본(09_...LightGBM cell 8)과 동일하게 test_size=0.2, random_state=0으로 분할한다 — 이 노트북 위쪽(§XGBoost~GradientBoost)에서 쓴 random_state=156, stratify=y_labels와는 다르다는 점에 유의한다.
# 업무적 의미: LightGBM 참고치(0.9582 등)를 그대로 재현하려면 분할 방식까지 원본과 똑같이 맞춰야 한다 — 분할이 다르면 같은 하이퍼파라미터를 써도 test셋 구성 자체가 달라 값이 달라진다.
X_train2, X_test2, y_train2, y_test2 = train_test_split(
    X_features_143, y_labels, test_size=0.2, random_state=0
)

# 기술적 의미: 원본(09_...LightGBM cell 10)과 동일하게, 학습셋(X_train2)을 다시 70/30으로 나눠 조기종료 전용 검증셋(X_val2)을 만든다.
# 업무적 의미: LightGBM·XGBoost처럼 조기종료를 쓰는 모델은 "학습에 전혀 쓰이지 않은 최종 테스트셋(X_test2)"과 "학습 중 조기종료 판단에만 쓰는 검증셋(X_val2)"을 분리해야 한다 — 테스트셋으로 조기종료를 판단하면 테스트 결과가 낙관적으로 왜곡된다(정보 누출).
X_tr2, X_val2, y_tr2, y_val2 = train_test_split(
    X_train2, y_train2, test_size=0.3, random_state=0
)

print(f"X_train2: {X_train2.shape}  X_test2: {X_test2.shape}")
print(f"X_tr2(최종 학습): {X_tr2.shape}  X_val2(조기종료 검증): {X_val2.shape}")


희소 피처(99% 이상 0) 제거: 163개 삭제
최종 피처 shape: (76020, 143)  (원본 09_...LightGBM 기준 143개와 일치해야 함)


X_train2: (60816, 143)  X_test2: (15204, 143)
X_tr2(최종 학습): (42571, 143)  X_val2(조기종료 검증): (18245, 143)


In [20]:
# [담당자: 정찬성 / 모델: LightGBM] 원본 HyperOpt 튜닝값 그대로 재현
# 왜: `09_산탄데르_고객_만족_LightGBM_정제후_metrics추가.ipynb` cell 14의 실행 결과(`best: {'learning_rate': 0.12554733872326435, 'max_depth': 113.0, 'min_child_samples': 83.0, 'num_leaves': 32.0, 'subsample': 0.9533039516434814}`)를 그대로 가져온다 — 이미 완료된 HyperOpt 탐색(TPE, 50회, rstate seed=30)을 다시 돌리지 않고, 이미 찾아 둔 최적값을 상수로 고정해 재사용한다.
from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.metrics import accuracy_score, roc_auc_score, recall_score, f1_score

# 기술적 의미: 원본 cell 15와 동일한 하이퍼파라미터로 LGBMClassifier를 생성한다. num_leaves/max_depth/min_child_samples는 int로, subsample/learning_rate는 원본과 동일하게 소수점 5자리로 반올림했다.
lgbm_orig = LGBMClassifier(
    n_estimators=500,
    num_leaves=32,
    max_depth=113,
    min_child_samples=83,
    subsample=round(0.9533039516434814, 5),
    learning_rate=round(0.12554733872326435, 5),
    random_state=156,
    verbose=-1,
)

# 기술적 의미: 원본과 동일하게 X_tr2/y_tr2로 학습하고, eval_set=[(X_tr2,y_tr2),(X_val2,y_val2)]로 조기종료 판단용 검증셋을 함께 전달한다. stopping_rounds=100은 원본의 early_stopping_rounds=100과 동일한 의미다.
lgbm_orig.fit(
    X_tr2, y_tr2,
    eval_set=[(X_tr2, y_tr2), (X_val2, y_val2)],
    eval_metric='auc',
    callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)],
)

# 기술적 의미: 최종 테스트셋(X_test2)에 대한 예측 클래스(기본 임계값 0.5)와 예측 확률을 구한다 — §09_...LightGBM cell 16과 동일한 평가 방식(임계값 조정 없음)이다.
y_pred_lgbm2 = lgbm_orig.predict(X_test2)
pred_proba_lgbm2 = lgbm_orig.predict_proba(X_test2)[:, 1]

# 기술적 의미: Accuracy/AUC/Recall/F1 4개 지표를 계산한다.
acc_lgbm2 = accuracy_score(y_test2, y_pred_lgbm2)
auc_lgbm2 = roc_auc_score(y_test2, pred_proba_lgbm2)
recall_lgbm2 = recall_score(y_test2, y_pred_lgbm2)
f1_lgbm2 = f1_score(y_test2, y_pred_lgbm2)

# 기술적 의미: §09_...LightGBM cell 16과 동일한 "[팀 결과표 기재용]" 형식으로 출력한다 — 오류화면1.png의 LightGBM 행(0.9582/0.8417/0.0016/0.0031)과 직접 비교하기 위함이다.
print('[팀 결과표 기재용] accuracy={0:.4f} | roc_auc={1:.4f} | recall={2:.4f} | f1={3:.4f}'.format(
    acc_lgbm2, auc_lgbm2, recall_lgbm2, f1_lgbm2))
print('(참고: 오류화면1.png 기록값 — accuracy=0.9582 | roc_auc=0.8417 | recall=0.0016 | f1=0.0031)')


[팀 결과표 기재용] accuracy=0.9582 | roc_auc=0.8417 | recall=0.0016 | f1=0.0031
(참고: 오류화면1.png 기록값 — accuracy=0.9582 | roc_auc=0.8417 | recall=0.0016 | f1=0.0031)


In [21]:
# [담당자: 정찬성 / 모델: XGBoost] 원본 파이프라인(143개 피처, 09_...LightGBM과 동일 분할) 기준 재평가
# 왜: §위 §Part XGBoost(cell 9)는 306개 피처·다른 분할(stratify=y_labels, random_state=156)로 학습했다 — LightGBM 참고치와 같은 기준으로 비교하려면 XGBoost도 143개 피처·원본 분할로 다시 학습해야 한다. 하이퍼파라미터는 §Part XGBoost(cell 9)와 동일하게 유지해, "피처·분할만 바뀌었을 때 XGBoost 성능이 어떻게 달라지는가"를 그 자체로도 확인할 수 있게 했다.
from xgboost import XGBClassifier

xgb_orig = XGBClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=5, min_child_weight=5,
    subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=100,
    eval_metric='auc', random_state=156,
)

# 기술적 의미: X_tr2/y_tr2로 학습하고, eval_set에 (X_tr2,y_tr2)와 (X_val2,y_val2)를 함께 전달해 검증셋 기준 조기종료를 적용한다.
xgb_orig.fit(
    X_tr2, y_tr2,
    eval_set=[(X_tr2, y_tr2), (X_val2, y_val2)],
    verbose=False,
)

# 기술적 의미: 최종 테스트셋(X_test2)에 기본 임계값 0.5로 예측 클래스와 확률을 구한다.
y_pred_xgb2 = xgb_orig.predict(X_test2)
pred_proba_xgb2 = xgb_orig.predict_proba(X_test2)[:, 1]

acc_xgb2 = accuracy_score(y_test2, y_pred_xgb2)
auc_xgb2 = roc_auc_score(y_test2, pred_proba_xgb2)
recall_xgb2 = recall_score(y_test2, y_pred_xgb2)
f1_xgb2 = f1_score(y_test2, y_pred_xgb2)

print('[팀 결과표 기재용] accuracy={0:.4f} | roc_auc={1:.4f} | recall={2:.4f} | f1={3:.4f}'.format(
    acc_xgb2, auc_xgb2, recall_xgb2, f1_xgb2))


[팀 결과표 기재용] accuracy=0.9583 | roc_auc=0.8450 | recall=0.0000 | f1=0.0000


In [22]:
# [담당자: 정찬성 / 모델: RandomForest] 원본 파이프라인(143개 피처) 기준 재평가
# 왜: 배깅 계열은 조기종료 개념이 없어 별도 검증셋(X_val2)이 필요 없다 — 학습셋 전체(X_train2, 80%)를 그대로 사용해 LightGBM/XGBoost(70%만 최종 학습에 사용)보다 더 많은 데이터로 학습한다는 차이가 있음에 유의한다.
from sklearn.ensemble import RandomForestClassifier

rf_orig = RandomForestClassifier(
    n_estimators=500, max_depth=5, min_samples_leaf=5,
    max_features=0.8, max_samples=0.8, n_jobs=-1, random_state=156,
)

# 기술적 의미: X_train2(80% 전체)로 학습한다.
rf_orig.fit(X_train2, y_train2)

y_pred_rf2 = rf_orig.predict(X_test2)
pred_proba_rf2 = rf_orig.predict_proba(X_test2)[:, 1]

acc_rf2 = accuracy_score(y_test2, y_pred_rf2)
auc_rf2 = roc_auc_score(y_test2, pred_proba_rf2)
recall_rf2 = recall_score(y_test2, y_pred_rf2)
f1_rf2 = f1_score(y_test2, y_pred_rf2)

print('[팀 결과표 기재용] accuracy={0:.4f} | roc_auc={1:.4f} | recall={2:.4f} | f1={3:.4f}'.format(
    acc_rf2, auc_rf2, recall_rf2, f1_rf2))


[팀 결과표 기재용] accuracy=0.9583 | roc_auc=0.8369 | recall=0.0000 | f1=0.0000


In [23]:
# [담당자: 정찬성 / 모델: LogisticRegression] 원본 파이프라인(143개 피처) 기준 재평가
# 왜: 선형모델도 조기종료가 필요 없어 X_train2(80%) 전체를 그대로 사용한다.
from sklearn.linear_model import LogisticRegression

lr_orig = LogisticRegression(max_iter=1000, random_state=156)
lr_orig.fit(X_train2, y_train2)

y_pred_lr2 = lr_orig.predict(X_test2)
pred_proba_lr2 = lr_orig.predict_proba(X_test2)[:, 1]

acc_lr2 = accuracy_score(y_test2, y_pred_lr2)
auc_lr2 = roc_auc_score(y_test2, pred_proba_lr2)
recall_lr2 = recall_score(y_test2, y_pred_lr2)
f1_lr2 = f1_score(y_test2, y_pred_lr2)

print('[팀 결과표 기재용] accuracy={0:.4f} | roc_auc={1:.4f} | recall={2:.4f} | f1={3:.4f}'.format(
    acc_lr2, auc_lr2, recall_lr2, f1_lr2))


[팀 결과표 기재용] accuracy=0.9583 | roc_auc=0.6195 | recall=0.0000 | f1=0.0000


In [24]:
# [담당자: 정찬성 / 모델: GradientBoost] 원본 파이프라인(143개 피처) 기준 재평가
# 왜: sklearn GradientBoostingClassifier는 별도 eval_set을 받지 않으므로, X_train2(80%) 전체를 넘기고 내부 validation_fraction(10%)으로 조기종료를 자체 판단하게 한다(n_iter_no_change=100) — §위 §Part GradientBoost와 동일한 방식이다.
from sklearn.ensemble import GradientBoostingClassifier

gb_orig = GradientBoostingClassifier(
    n_estimators=500, learning_rate=0.05, max_depth=5, subsample=0.8,
    random_state=156, n_iter_no_change=100, validation_fraction=0.1,
)
gb_orig.fit(X_train2, y_train2)

y_pred_gb2 = gb_orig.predict(X_test2)
pred_proba_gb2 = gb_orig.predict_proba(X_test2)[:, 1]

acc_gb2 = accuracy_score(y_test2, y_pred_gb2)
auc_gb2 = roc_auc_score(y_test2, pred_proba_gb2)
recall_gb2 = recall_score(y_test2, y_pred_gb2)
f1_gb2 = f1_score(y_test2, y_pred_gb2)

print('[팀 결과표 기재용] accuracy={0:.4f} | roc_auc={1:.4f} | recall={2:.4f} | f1={3:.4f}'.format(
    acc_gb2, auc_gb2, recall_gb2, f1_gb2))


[팀 결과표 기재용] accuracy=0.9578 | roc_auc=0.8400 | recall=0.0079 | f1=0.0154


### [담당자: 정찬성] 최종 종합 — 오류화면1.png 스프레드시트 채우기

```mermaid
flowchart LR
    A["5개 모델(143개 피처, 원본 분할 기준)<br/>Accuracy/AUC/Recall/F1"] --> B["오류화면1.png와 동일한<br/>행 순서(LightGBM→로지스틱회귀→XGBoost→랜덤포레스트→GradientBoost)로 표 작성"]
```

In [25]:
# [담당자: 정찬성] 오류화면1.png 표와 동일한 행 순서·열 구성으로 최종 결과를 정리한다.
final_team_table = pd.DataFrame([
    {'모델': 'LightGBM', 'Accuracy': acc_lgbm2, 'AUC': auc_lgbm2, 'Recall': recall_lgbm2, 'F1': f1_lgbm2},
    {'모델': '로지스틱 회귀', 'Accuracy': acc_lr2, 'AUC': auc_lr2, 'Recall': recall_lr2, 'F1': f1_lr2},
    {'모델': 'XGBoost', 'Accuracy': acc_xgb2, 'AUC': auc_xgb2, 'Recall': recall_xgb2, 'F1': f1_xgb2},
    {'모델': '랜덤포레스트', 'Accuracy': acc_rf2, 'AUC': auc_rf2, 'Recall': recall_rf2, 'F1': f1_rf2},
    {'모델': 'GradientBoost', 'Accuracy': acc_gb2, 'AUC': auc_gb2, 'Recall': recall_gb2, 'F1': f1_gb2},
])

print('=== 오류화면1.png 표 형식 — 5개 모델 최종 결과(143개 피처, 원본 분할 기준, 임계값 0.5) ===')
print(final_team_table.to_string(index=False))


=== 오류화면1.png 표 형식 — 5개 모델 최종 결과(143개 피처, 원본 분할 기준, 임계값 0.5) ===
           모델  Accuracy      AUC   Recall       F1
     LightGBM  0.958235 0.841748 0.001577 0.003140
      로지스틱 회귀  0.958300 0.619491 0.000000 0.000000
      XGBoost  0.958300 0.844977 0.000000 0.000000
       랜덤포레스트  0.958300 0.836929 0.000000 0.000000
GradientBoost  0.957840 0.839995 0.007886 0.015361


### 결과 해석 및 검증

> ⏸ 이 셀은 실행 대기 상태다 — 위 코드 셀들을 순서대로 실행한 뒤, 아래 항목을 실제 출력값으로 채운다.

- **재현 검증(최우선 확인 사항)**: LightGBM 행이 `오류화면1.png`의 기록값(accuracy=0.9582 | roc_auc=0.8417 | recall=0.0016 | f1=0.0031)과 소수점 몇 자리까지 일치하는지 확인할 것. 완전히 일치한다면 이 절의 143개 피처·분할·하이퍼파라미터 재현이 정확했다는 뜻이고, 값이 다르다면 LightGBM 라이브러리 버전 차이(트리 분할 결과가 버전에 따라 미세하게 달라질 수 있음) 등을 원인으로 검토해야 한다.
- **XGBoost의 피처·분할 민감도**: §Part XGBoost(cell 9, 306개 피처·stratify 분할)의 실측치(AUC 0.8520)와 이 절의 XGBoost(143개 피처·원본 분할) 결과를 비교해, 트리 기반 모델이 피처 개수·분할 방식 변화에 얼마나 민감한지 확인할 것.
- **팀 스프레드시트 기재**: `final_team_table`의 값을 그대로 `오류화면1.png` 스프레드시트의 로지스틱회귀/XGBoost/랜덤포레스트/GradientBoost 빈 칸에 옮겨 적으면 된다 — 5개 모델 모두 정확히 같은 143개 피처·같은 분할·같은 임계값(0.5) 기준이므로 공정하게 비교 가능한 값이다.

---

## [담당자: 정찬성] 임계값 조정 — 원본 파이프라인(143개 피처) 5개 모델 재평가

### 왜 임계값 0.5가 부적절한가 (20년차 관점)

§바로 위 5개 모델(LightGBM/로지스틱회귀/XGBoost/랜덤포레스트/GradientBoost)을 기본 임계값 0.5로 평가한 결과, **3개 모델(로지스틱회귀/XGBoost/랜덤포레스트)의 Recall·F1이 정확히 0.0000**이었다. 점검 결과:

- **XGBoost(AUC 0.8450, 5개 중 1위)**와 **랜덤포레스트(AUC 0.8369)**는 판별력(AUC) 자체는 준수하지만, TARGET=1 비율이 약 3.96%인 극단적 불균형 데이터에서는 로그손실/AUC로 학습된 확률이 대부분 사전확률(≈0.04) 근처에 몰려, 테스트셋 그 어떤 표본도 0.5를 넘기지 못하는 것이 전혀 이상하지 않다 — **모델이 아니라 "0.5라는 임계값 선택"이 이 데이터에 맞지 않는 것**이다.
- 이런 극단적 불균형(사전확률 ≪ 0.5) 상황에서 **분류 임계값을 사전확률(기저율)로 낮추는 것**은 불균형 분류에서 널리 쓰이는 표준적인 실무 휴리스틱이다 — 베이즈 최적 결정 경계가 사전확률 쪽으로 이동한다는 통계적 근거가 있고, `오류화면1.png`의 "임계값 0.04" 메모도 원래 이 사전확률(§09_...LightGBM 로그의 `pavg=0.038947`)을 가리키는 것이었다(§위 확인).
- 로지스틱회귀는 AUC(0.6195)가 트리 계열보다 훨씬 낮아 스케일링 미비라는 별도 원인이 있지만, 그 문제와 별개로 **동일한 임계값 기준으로 재평가**해 다른 4개 모델과 나란히 비교할 수 있게 한다.

이 절에서는 **모델을 다시 학습하지 않고**(§위에서 이미 학습된 `lgbm_orig`/`xgb_orig`/`rf_orig`/`lr_orig`/`gb_orig` 객체를 그대로 재사용), 분류 임계값만 `y_train2`의 TARGET=1 비율(≈0.04)로 낮춰 Accuracy/AUC/Recall/F1을 다시 계산한다. AUC는 임계값과 무관한 지표이므로 §위 표와 값이 같아야 하고, Accuracy/Recall/F1만 달라진다.

```mermaid
flowchart TD
    A["y_train2.mean() = 양성(TARGET=1) 기저율<br/>≈ 0.0396"] --> B["threshold2 = 기저율로 고정"]
    C["§위에서 이미 학습된 5개 모델<br/>(lgbm_orig/xgb_orig/rf_orig/lr_orig/gb_orig)"] --> D["X_test2에 대한 predict_proba() 재사용<br/>(재학습 없음)"]
    B --> E["pred = (proba >= threshold2)"]
    D --> E
    E --> F["Accuracy/Recall/F1 재계산<br/>(AUC는 임계값과 무관, §위 표와 동일)"]
    F --> G["오류화면1.png와 동일한 행 순서로<br/>'임계값 조정 후' 표 출력"]

    style G fill:#E0ECFF,stroke:#2E6DE5
```

In [26]:
# [담당자: 정찬성] 원본 파이프라인 5개 모델 — 기저율 임계값 재평가 (재학습 없음)
# 왜: §점검 결과, XGBoost/랜덤포레스트/로지스틱회귀 3개 모델의 Recall·F1이 임계값 0.5에서 정확히 0.0000이었다. 이는 극단적 불균형(TARGET=1 ≈ 3.96%) 데이터에서 흔히 나타나는 "임계값-사전확률 미스매치"이며, 재학습이 아니라 분류 임계값만 조정하면 되는 문제다.

# 기술적 의미: 학습 데이터(y_train2)에서 TARGET=1 비율을 구한다 — §위 §임계값 재평가 절(구 파이프라인)과 동일한 원리이나, 이번에는 143개 피처·원본 분할(y_train2)의 실제 비율을 다시 계산한다(구 y_train과 정확히 같은 값은 아닐 수 있음에 유의).
threshold2 = y_train2.mean()
print(f'분류 임계값(양성 클래스 기저율 기준, 원본 파이프라인): {threshold2:.4f}')


# 기술적 의미: 이미 학습된 모델·검증 데이터·임계값을 받아 Accuracy/AUC/Recall/F1 4개 지표를 dict로 계산해 반환하는 함수를 정의한다 — §최종 종합(cell 31)의 4개 지표(Precision 없이 Accuracy/AUC/Recall/F1)와 동일한 구성으로 맞췄다.
# 업무적 의미: model.predict()의 내장 임계값(0.5) 대신, predict_proba() 확률값에 threshold2를 직접 적용해 재분류한다.
def evaluate_at_threshold2(model, X_te, y_te, thr):
    # 기술적 의미: 검증 데이터에 대한 양성(TARGET=1) 확률을 구한다.
    pred_proba = model.predict_proba(X_te)[:, 1]
    # 기술적 의미: 확률이 임계값 이상이면 1, 미만이면 0으로 재분류한다.
    pred_at_threshold = (pred_proba >= thr).astype(int)
    # 기술적 의미: Accuracy/AUC/Recall/F1과 사용한 임계값을 dict로 반환한다. AUC는 임계값과 무관하게 확률값 자체로 계산되므로, §위 §최종 종합(cell 31)의 AUC와 정확히 같아야 한다.
    return {
        'Accuracy': accuracy_score(y_te, pred_at_threshold),
        'AUC': roc_auc_score(y_te, pred_proba),
        'Recall': recall_score(y_te, pred_at_threshold, zero_division=0),
        'F1': f1_score(y_te, pred_at_threshold, zero_division=0),
        '임계값': thr,
    }


# 기술적 의미: §위에서 이미 학습해 둔 5개 모델 객체(lgbm_orig/xgb_orig/rf_orig/lr_orig/gb_orig)에 대해 evaluate_at_threshold2()를 호출한다 — fit()을 다시 호출하지 않으므로 재학습 비용이 전혀 없다.
threshold2_results = {
    'LightGBM': evaluate_at_threshold2(lgbm_orig, X_test2, y_test2, threshold2),
    '로지스틱 회귀': evaluate_at_threshold2(lr_orig, X_test2, y_test2, threshold2),
    'XGBoost': evaluate_at_threshold2(xgb_orig, X_test2, y_test2, threshold2),
    '랜덤포레스트': evaluate_at_threshold2(rf_orig, X_test2, y_test2, threshold2),
    'GradientBoost': evaluate_at_threshold2(gb_orig, X_test2, y_test2, threshold2),
}

# 기술적 의미: dict of dict를 DataFrame으로 변환(전치)하고, 오류화면1.png와 동일한 행 순서로 재정렬한다.
final_team_table_adjusted = pd.DataFrame(threshold2_results).T
final_team_table_adjusted = final_team_table_adjusted.loc[
    ['LightGBM', '로지스틱 회귀', 'XGBoost', '랜덤포레스트', 'GradientBoost']
]

print('\n=== 임계값 조정 후 (기저율 기준) — 오류화면1.png 표 형식 ===')
print(final_team_table_adjusted.to_string())

# 기술적 의미: 임계값 0.5 기준 표(final_team_table, §cell 31)와 나란히 비교할 수 있도록 함께 출력한다.
print('\n=== (비교용) 임계값 0.5 기준 — §cell 31 재출력 ===')
print(final_team_table.to_string(index=False))


분류 임계값(양성 클래스 기저율 기준, 원본 파이프라인): 0.0390



=== 임계값 조정 후 (기저율 기준) — 오류화면1.png 표 형식 ===
               Accuracy       AUC    Recall        F1       임계값
LightGBM       0.779663  0.841748  0.744479  0.219842  0.039036
로지스틱 회귀        0.568600  0.619491  0.585174  0.101630  0.039036
XGBoost        0.769797  0.844977  0.763407  0.216652  0.039036
랜덤포레스트         0.737832  0.836929  0.771293  0.197019  0.039036
GradientBoost  0.797882  0.839995  0.717666  0.228471  0.039036

=== (비교용) 임계값 0.5 기준 — §cell 31 재출력 ===
           모델  Accuracy      AUC   Recall       F1
     LightGBM  0.958235 0.841748 0.001577 0.003140
      로지스틱 회귀  0.958300 0.619491 0.000000 0.000000
      XGBoost  0.958300 0.844977 0.000000 0.000000
       랜덤포레스트  0.958300 0.836929 0.000000 0.000000
GradientBoost  0.957840 0.839995 0.007886 0.015361


### 결과 해석

> ⏸ 이 셀은 실행 대기 상태다 — 위 코드 셀을 실행한 뒤, 아래 항목을 실제 출력값으로 채운다.

- **AUC가 §임계값 0.5 표(cell 31)와 정확히 같은지** 확인할 것 — AUC는 임계값과 무관한 지표이므로, 값이 다르다면 코드 상 오류(예: 다른 모델 객체 참조)를 의심해야 한다.
- **3개 모델(로지스틱회귀/XGBoost/랜덤포레스트)의 Recall·F1이 0.0000에서 벗어났는지** — §점검에서 예상한 대로 XGBoost·랜덤포레스트는 AUC가 준수했으므로(0.84 내외) 임계값만 낮춰도 유의미한 Recall이 나올 가능성이 높다.
- **로지스틱회귀는 임계값을 낮춰도 여전히 성능이 낮을 가능성**이 크다 — AUC 자체가 0.6195로 낮아(스케일링 미비가 원인으로 추정) 임계값 조정만으로는 근본적인 판별력 문제가 해결되지 않는다. 이 모델은 별도로 `StandardScaler` 적용을 검토할 대상으로 남겨 둔다(이번 작업 범위에는 포함하지 않음).
- **최종 팀 스프레드시트(`오류화면1.png`) 기재 시**: Accuracy/AUC/Recall/F1 중 어느 임계값 기준 값을 채울지는 "정찬성" 개인의 §참고 행(LightGBM, 임계값 0.5, 0.9582/0.8417/0.0016/0.0031)과 형식을 맞출지, 아니면 임계값 조정 후 값으로 전원 통일할지 팀 내 합의가 필요한 사항이다 — 이 노트북은 두 버전(§cell 31 임계값 0.5, §바로 위 임계값 조정 후)을 모두 남겨 두어 어느 쪽이든 선택해 옮겨 적을 수 있게 했다.